# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byte-AbhijitVichare/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

## Unit of Analysis

For this analysis, I use the FlyRank warehouse sample table for June 2026 to understand the data contract and verify the dataset structure. The sample table represents the final month and is used only for exploration, not for developing prediction labels.

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

## Field Categories

### Features
- impressions
- clicks
- average position
- CTR
- engagement rate

These features are available before the refresh decision and describe historical search performance.

### Label
- needs_refresh

This proxy label indicates whether a page experienced a significant decline in impressions and should be reviewed.

### Context
- content_type
- content_age_days
- client_hash_id

These variables provide additional context but are not direct performance signals.

### Excluded
- trend_direction
- trend_pct

These fields are derived from future performance and would introduce data leakage if used during training.

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [42]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [43]:
grain = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT content_hash_id) AS unique_pages
FROM {TABLES['fact_daily_sample']}
""").df()

grain


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,unique_pages
0,11694072,409205


In [44]:
dates = con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM {TABLES['fact_daily_sample']}
""").df()

dates

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows
0,2026-06-01,2026-06-30,11694072


In [45]:
availability = con.sql(f"""
SELECT
COUNT(*) AS available_rows
FROM {TABLES['fact_daily_sample']}
WHERE gsc_impressions IS NOT NULL
AND (gsc_impressions > 0) IS TRUE
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3878937


## 4. Data limits

## Data Limitations

This dataset is intended to support decision-making rather than prove causal relationships.

The history available differs across clients, resulting in an unbalanced panel.

Some clients only have Google Search Console data for part of the available timeline.

The sample table represents only the final month and should not be used for developing prediction labels.

Recommendations generated from this analysis should always be reviewed by a human before action is taken.

In [46]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.